In [ ]:
import os
import joblib
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X_train = pd.read_csv(f"../data/processed/X_train_processed.csv")
y_train = pd.read_csv(f"../data/processed/y_train_processed.csv").values.ravel()

In [10]:
"""
Step 4: Hyperparameter Tuning for Top Contenders
Description: จูนพารามิเตอร์ให้กับโมเดลกลุ่มผู้นำ [LightGBM + SMOTE] และ [XGBoost + Baseline]
             แยกการทำงานออกเป็นบล็อกเพื่อความสะอาดและง่ายต่อการบำรุงรักษาโค้ด
"""

import os
import joblib
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

# =====================================================================
# BLOCK 1: GLOBAL CONFIGURATION & DATA LOADING
# =====================================================================
print("[BLOCK 1] Loading processed dataset...")

# Path ข้อมูลและ Cross-Validation Setup
DATA_DIR = "../data/processed"
MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

X_train = pd.read_csv(f"{DATA_DIR}/X_train_processed.csv")
y_train = pd.read_csv(f"{DATA_DIR}/y_train_processed.csv").values.ravel()

# ใช้ StratifiedKFold 5-Fold ตามเกณฑ์การทดลอง
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"-> Data loaded. Shapes: X_train {X_train.shape}, y_train {y_train.shape}")


# =====================================================================
# BLOCK 2: TUNING CONFIGURATION FOR LIGHTGBM + SMOTE
# =====================================================================
print("\n[BLOCK 2] Configuring Hyperparameter Tuning for LightGBM + SMOTE...")

# สร้าง Pipeline ที่รวม SMOTE และ LightGBM เข้าด้วยกัน
lgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1, num_leaves=31))
])

# Parameter Grid สำหรับ LightGBM (ระบุชื่อ 'clf__' นำหน้าพารามิเตอร์ของโมเดล)
lgb_param_grid = {
    'clf__n_estimators':[100, 200, 300],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__max_depth': [3, 5, 7]
}

# นิยาม GridSearchCV สำหรับ LightGBM
lgb_search = GridSearchCV(
    estimator=lgb_pipeline,
    param_grid=lgb_param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)


# =====================================================================
# BLOCK 3: TUNING CONFIGURATION FOR XGBOOST + BASELINE
# =====================================================================
print("\n[BLOCK 3] Configuring Hyperparameter Tuning for XGBoost + Baseline...")

# สร้างโมเดล XGBoost โดยตรง (ไม่ต้องใช้ Pipeline เนื่องจากผลทดสอบไม่ต้องพึ่งพา SMOTE)
xgb_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss')

# Parameter Grid สำหรับ XGBoost
xgb_param_grid = {
    'n_estimators':[100, 200, 300] ,
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth':[3, 5, 7],
    'subsample': [0.8, 1.0]
}

# นิยาม GridSearchCV สำหรับ XGBoost
xgb_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)


# =====================================================================
# BLOCK 4: EXECUTION (TUNING PROCESS)
# =====================================================================
print("\n[BLOCK 4] Running execution process...")

# รันการจูนโมเดลตัวที่ 1
print("Executing LightGBM Tuning...")
lgb_search.fit(X_train, y_train)
print(f"-> LightGBM Best CV F1-Macro: {lgb_search.best_score_:.6f}")

# รันการจูนโมเดลตัวที่ 2
print("\nExecuting XGBoost Tuning...")
xgb_search.fit(X_train, y_train)
print(f"-> XGBoost Best CV F1-Macro: {xgb_search.best_score_:.6f}")


# =====================================================================
# BLOCK 5: MODEL COMPARISON & SERIALIZATION (SAVE MODEL)
# =====================================================================
print("\n[BLOCK 5] Comparing results and saving the best model...")

print("=" * 60)
# เปรียบเทียบคะแนนเพื่อหาผู้ชนะที่แท้จริงหลังการจูน
if lgb_search.best_score_ > xgb_search.best_score_:
    print(f"🏆 FINAL WINNER: LightGBM + SMOTE | CV F1-Macro: {lgb_search.best_score_:.6f}")
    print(f"Best Hyperparameters: {lgb_search.best_params_}")
    final_model = lgb_search.best_estimator_
else:
    print(f"🏆 FINAL WINNER: XGBoost + Baseline | CV F1-Macro: {xgb_search.best_score_:.6f}")
    print(f"Best Hyperparameters: {xgb_search.best_params_}")
    final_model = xgb_search.best_estimator_
print("=" * 60)

# บันทึกโมเดลที่ดีที่สุดลง Disk เพื่อส่งต่อไปยัง Step 5
output_path = f"{MODEL_DIR}/final_best_disaster_model.pkl"
joblib.dump(final_model, output_path)

print(f"Successfully serialized and saved the best model to: '{output_path}'")


[BLOCK 1] Loading processed dataset...
-> Data loaded. Shapes: X_train (13979, 39), y_train (13979,)

[BLOCK 2] Configuring Hyperparameter Tuning for LightGBM + SMOTE...

[BLOCK 3] Configuring Hyperparameter Tuning for XGBoost + Baseline...

[BLOCK 4] Running execution process...
Executing LightGBM Tuning...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
-> LightGBM Best CV F1-Macro: 0.912301

Executing XGBoost Tuning...
Fitting 5 folds for each of 54 candidates, totalling 270 fits
-> XGBoost Best CV F1-Macro: 0.913050

[BLOCK 5] Comparing results and saving the best model...
🏆 FINAL WINNER: XGBoost + Baseline | CV F1-Macro: 0.913050
Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}
Successfully serialized and saved the best model to: '../models/final_best_disaster_model.pkl'
